# 11 — Sismo en tiempo real

## 1. Setup — imports

In [ ]:
import glob
import json
import os
import time
import uuid
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import RegularGridInterpolator
from scipy.signal import butter, cont2discrete, detrend, lfilter, sosfilt_zi, sosfiltfilt, ss2tf
from scipy.signal.windows import tukey
from scipy.stats import kurtosis

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

## 2. Catálogo, esquemas y parámetros

In [ ]:
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"
TR = f"{catalogo}.tiempo_real"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TR} COMMENT 'Simulación de sismos en tiempo real (notebook 11)'")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

In [ ]:
SEED = 700
FILTRO_HZ = (0.1, 25.0)
AMORTIGUAMIENTO = 0.05
PERIODOS = np.round(np.geomspace(0.05, 4.0, 50), 4)
G_CM_S2 = 980.665
VS30_GRID = np.arange(150.0, 1001.0, 10.0)
EVENTO_EFECTO = "2025-06-15_callao_M6.1"


def referencia(tipo):
    return pd.DataFrame([json.loads(r.valor) for r in spark.table("gold_referencias").where(F.col("tipo") == tipo).select("valor").collect()])


REG_LIMA = spark.table("gold_registros").where("ciudad = 'Lima'")
dist = referencia("distritos")

## 3. Funciones: escritura, señal y métricas

In [ ]:
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append((t.tableName, spark.table(t.tableName).count(), round((d["sizeInBytes"] or 0) / 1e6, 1)))
    return spark.createDataFrame(filas, "tabla string, filas long, mb double")

In [ ]:
_CACHE_SDOF = {}


def _filtro_sdof(periodo, dt, z):
    if (periodo, dt, z) in _CACHE_SDOF:
        return _CACHE_SDOF[(periodo, dt, z)]
    w = 2 * np.pi / periodo
    A = np.array([[0.0, 1.0], [-w * w, -2 * z * w]])
    B = np.array([[0.0], [-1.0]])
    C = np.array([[1.0, 0.0]])
    D = np.array([[0.0]])
    Ad, Bd, Cd, Dd, _ = cont2discrete((A, B, C, D), dt, method="foh")
    b, a = ss2tf(Ad, Bd, Cd, Dd)
    _CACHE_SDOF[(periodo, dt, z)] = (b[0], a, w)
    return _CACHE_SDOF[(periodo, dt, z)]


def espectro_sa(acc_g, dt, periodos=PERIODOS, z=AMORTIGUAMIENTO):
    sa = np.empty(len(periodos))
    for k, periodo in enumerate(periodos):
        b, a, w = _filtro_sdof(float(periodo), float(dt), z)
        sa[k] = w * w * np.abs(lfilter(b, a, acc_g)).max()
    return sa


def filtrar_cms2(acc_cms2, fs):
    a = detrend(acc_cms2 - acc_cms2.mean(), type="linear")
    a = a * tukey(len(a), 0.05)
    sos = butter(4, FILTRO_HZ, btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, a)


def procesar_componente(acc_cms2, fs):
    dt = 1.0 / fs
    a = filtrar_cms2(acc_cms2, fs)
    vel = cumulative_trapezoid(a, dx=dt, initial=0.0)
    acc_g = a / G_CM_S2
    return acc_g, np.abs(acc_g).max(), np.abs(vel).max(), espectro_sa(acc_g, dt)


def intensidad_arias(acc_g, fs):
    return np.pi / (2 * 9.80665) * np.cumsum((acc_g * 9.80665) ** 2) / fs


def duracion_d595(arias, fs):
    return (np.searchsorted(arias, 0.95 * arias[-1]) - np.searchsorted(arias, 0.05 * arias[-1])) / fs


def variables_onda(x1, x2, fs):
    k = min(len(x1), len(x2))
    pgas, pgvs, sas, arias, d595 = [], [], [], [], []
    for x in (x1[:k], x2[:k]):
        acc_g, pga, pgv, sa = procesar_componente(np.asarray(x, float), fs)
        ia = intensidad_arias(acc_g, fs)
        pgas.append(pga); pgvs.append(pgv); sas.append(sa); arias.append(ia[-1]); d595.append(duracion_d595(ia, fs))
    sa_geo = np.sqrt(sas[0] * sas[1])
    return {"pga_geo_g": float(np.sqrt(pgas[0] * pgas[1])), "pgv_geo_cms": float(np.sqrt(pgvs[0] * pgvs[1])),
            "tp_s": float(PERIODOS[int(sa_geo.argmax())]), "sa_max_geo_g": float(sa_geo.max()), "sa_geo": sa_geo.tolist(),
            "arias_ms": float(np.sqrt(arias[0] * arias[1])), "d595_s": float(np.sqrt(d595[0] * d595[1]))}

In [ ]:
METRICAS = []


def registrar_modelo(nombre_modelo, params, metricas):
    for k, v in metricas.items():
        METRICAS.append({"modelo": nombre_modelo, "metrica": k, "valor": float(v), "parametros": json.dumps(params, default=str)})
    print(f"  {nombre_modelo}: " + " | ".join(f"{k} = {v:,.4f}" for k, v in metricas.items()))


def guardar_metricas(notebook):
    pdf = pd.DataFrame(METRICAS, columns=["modelo", "metrica", "valor", "parametros"])
    pdf["fecha"] = datetime.now().isoformat(timespec="seconds")
    pdf["notebook"] = notebook
    (a_spark(pdf, "modelo string, metrica string, valor double, parametros string, fecha string, notebook string")
     .write.mode("overwrite").option("replaceWhere", f"notebook = '{notebook}'").saveAsTable("ml_metricas"))
    display(pdf)


def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))

## 4. Champions de onda P, amplificación y daño; cadena de demanda guardada por 07

In [ ]:
def champion(caso):
    fila = (spark.table("ml_registro_modelos").filter((F.col("caso") == caso) & F.col("promovido"))
            .orderBy(F.col("version").desc()).first())
    if fila is None:
        raise RuntimeError(f"No hay champion de {caso}: ejecute antes su notebook de ML")
    meta = {"version": int(fila["version"]), "algoritmo": fila["algoritmo"], "lote": fila["lote"],
            "ruta": fila["ruta"], "cols_num": json.loads(fila["columnas_num"]), "metricas": json.loads(fila["metricas"]), "detalle": json.loads(fila["detalle"])}
    print(f"Champion de {caso}: v{meta['version']} ({meta['algoritmo']}, lote {meta['lote']})")
    return PipelineModel.load(fila["ruta"]), meta


modelo_dano, meta_dano = champion("dano")
modelo_onda, meta_onda = champion("clasificacion")

In [ ]:
VARS_ONDA = ["PGA_g", "PGV_cms", "Sa03_g", "Sa06_g", "Sa10_g", "Sa_Tb"]
RANGO_DANO = meta_dano["detalle"]["rangos"]


def variables_lima(df):
    return (df.withColumn("PGV_cms", F.col("pgv_cms"))
            .withColumn("suelo_cod", F.when(F.col("vs30") < 180, 3.0).when(F.col("vs30") < 500, 2.0).when(F.col("vs30") < 1500, 1.0).otherwise(0.0)))


def fuera_de_rango(columna):
    a, b = RANGO_DANO[columna]
    return ~F.col(columna).between(a, b)


def dano_edificios(df):
    pred = modelo_dano.transform(variables_lima(df)).withColumn("indice_dano", vector_to_array("probability")[1])
    return dano_gem(pred.drop("probability", "rawPrediction", "prediction"))


def efecto_por_distrito(pred, claves):
    return (pred.groupBy(*claves).agg(
        F.count("*").alias("edificios"),
        F.expr("percentile_approx(PGA_g, 0.5)").alias("PGA_mediana_g"),
        F.expr("percentile_approx(Sa_Tb, 0.5)").alias("Sa_Tb_mediana_g"),
        F.avg("indice_dano").alias("indice_dano_medio"),
        (100 * F.avg((F.col("indice_dano") > 0.5).cast("double"))).alias("pct_indice_mayor_05"),
        F.avg("P_ge_severo").alias("P_ge_severo_medio"), F.avg("P_ge_colapso").alias("P_colapso_medio"),
        F.sum("P_ge_severo").alias("edificios_severo_esperados"),
        *[(100 * F.avg(fuera_de_rango(c).cast("double"))).alias(f"pct_fuera_rango_{c}") for c in VARS_ONDA if c in RANGO_DANO]))

In [ ]:
PISOS_CONCRETO, PISOS_MAX_LIVIANO = 5, 2
ESTADOS = ["leve", "moderado", "severo", "colapso"]
FRAG = spark.table("riesgo_fragilidad_parametros").drop("fuente", "taxonomia_gem")
PISOS_MAX_GEM = {r["tipologia"]: int(r["tope"]) for r in FRAG.groupBy("tipologia").agg(F.max("pisos_gem").alias("tope")).collect()}
CENSO = spark.table("gold_edificios").select("ubigeo", "p_adobe", "p_quincha", "p_madera").distinct()


def phi(x):
    z = F.abs(x) / np.sqrt(2.0)
    t = 1 / (1 + 0.3275911 * z)
    poli = t * (0.254829592 + t * (-0.284496736 + t * (1.421413741 + t * (-1.453152027 + t * 1.061405429))))
    erf = 1 - poli * F.exp(-z * z)
    return F.when(x >= 0, 0.5 * (1 + erf)).otherwise(0.5 * (1 - erf))


def tipologia_gem():
    u = (F.abs(F.xxhash64("id")) % 1_000_000) / 1_000_000.0
    liviano = F.col("pisos") <= PISOS_MAX_LIVIANO
    return (F.when(F.col("pisos") >= PISOS_CONCRETO, "concreto_armado")
            .when(liviano & (u < F.col("p_adobe")), "adobe")
            .when(liviano & (u < F.col("p_adobe") + F.col("p_quincha")), "quincha")
            .when(liviano & (u < F.col("p_adobe") + F.col("p_quincha") + F.col("p_madera")), "madera")
            .otherwise("albanileria_confinada"))


def tope_gem():
    tope = F.lit(PISOS_MAX_GEM["albanileria_confinada"])
    for tip, t in PISOS_MAX_GEM.items():
        tope = F.when(F.col("tipologia") == tip, F.lit(t)).otherwise(tope)
    return tope


def dano_gem(df):
    im_valor = (F.when(F.col("im") == "PGA", F.col("PGA_g")).when(F.col("im") == "SA(0.3s)", F.col("Sa03_g"))
                .when(F.col("im") == "SA(0.6s)", F.col("Sa06_g")).otherwise(F.col("Sa10_g")))
    d = (df.join(F.broadcast(CENSO), "ubigeo", "left").fillna(0.0, subset=["p_adobe", "p_quincha", "p_madera"])
         .withColumn("tipologia", tipologia_gem()).withColumn("pisos_gem", F.least(F.col("pisos"), tope_gem()))
         .join(F.broadcast(FRAG), ["tipologia", "pisos_gem"]).withColumn("IM_g", im_valor))
    for e in ESTADOS:
        d = d.withColumn(f"P_ge_{e}", phi(F.log(F.col("IM_g") / F.col(f"theta_{e}")) / F.col("beta")))
    return d

In [ ]:
param = {r["clave"]: json.loads(r["valor"]) for r in spark.table("riesgo_parametros").collect()}
amp = spark.table("riesgo_amplificacion_lima").select("vs30", "indice_periodo", "ln_F", "F").toPandas().sort_values(["vs30", "indice_periodo"])
ln_f = amp.ln_F.to_numpy().reshape(len(VS30_GRID), len(PERIODOS))
F_GRID = amp.F.to_numpy().reshape(len(VS30_GRID), len(PERIODOS))
interp_ln_f = RegularGridInterpolator((VS30_GRID, np.log(PERIODOS)), ln_f)


def ln_amplificacion(T, vs30):
    T, vs30 = np.broadcast_arrays(np.asarray(T, float), np.asarray(vs30, float))
    puntos = np.column_stack([np.clip(vs30.ravel(), VS30_GRID[0], VS30_GRID[-1]),
                              np.log(np.clip(T.ravel(), PERIODOS[0], PERIODOS[-1]))])
    return interp_ln_f(puntos).reshape(T.shape)


PGA_REF_NL = float(param["pga_ref_nl"])
nl = referencia("bssa14").astype(float)
nl = nl[nl.T_s > 0].sort_values("T_s")
F4_NL = np.interp(np.log(PERIODOS), np.log(nl.T_s), nl.f4)
F5_NL = np.interp(np.log(PERIODOS), np.log(nl.T_s), nl.f5)


def ln_F(periodo, vs30, pga_r):
    periodo, vs30, pga_r = np.broadcast_arrays(np.asarray(periodo, float), np.asarray(vs30, float), np.asarray(pga_r, float))
    f4 = np.interp(np.log(periodo), np.log(PERIODOS), F4_NL)
    f5 = np.interp(np.log(periodo), np.log(PERIODOS), F5_NL)
    return ln_amplificacion(periodo, vs30) + f4 * (np.exp(f5 * (np.minimum(vs30, 760.0) - 360.0)) - np.exp(f5 * 400.0)) * (np.log((pga_r + 0.1) / 0.1) - np.log((PGA_REF_NL + 0.1) / 0.1))


tb = np.array([r["Tb"] for r in spark.table("gold_edificios").select("Tb").distinct().collect()], float)
i = np.clip(np.searchsorted(PERIODOS, tb) - 1, 0, len(PERIODOS) - 2)
w = np.clip((np.log(tb) - np.log(PERIODOS[i])) / (np.log(PERIODOS[i + 1]) - np.log(PERIODOS[i])), 0, 1)
tb_tab = a_spark(pd.DataFrame({"Tb": tb, "idx": i + 1, "w": w}), "Tb double, idx int, w double")
amp_arr = a_spark(pd.DataFrame({"vs30_bin": VS30_GRID, "F_arr": [fila.tolist() for fila in F_GRID]}), "vs30_bin double, F_arr array<double>")
sitio_arr = spark.table("riesgo_sitio_h3")
F_SITIO = F.expr("zip_with(F_arr, coalesce(dS_arr, array_repeat(CAST(1.0 AS DOUBLE), size(F_arr))), (a, b) -> a * b)")
arr_sql = lambda v: "array(" + ", ".join(f"{x:.6f}D" for x in v) + ")"
F_NO_LINEAL = F.expr(f"transform(F_arr, (x, i) -> x * exp({arr_sql(F4_NL)}[i] * (exp({arr_sql(F5_NL)}[i] * (least(vs30, 760D) - 360D)) "
                     f"- exp({arr_sql(F5_NL)}[i] * 400D)) * (ln((pga_r_nl + 0.1D) / 0.1D) - {np.log((PGA_REF_NL + 0.1) / 0.1):.6f}D)))")


def interp_log(arreglo):
    return F.exp((1 - F.col("w")) * F.log(F.expr(f"element_at({arreglo}, idx)"))
                 + F.col("w") * F.log(F.expr(f"element_at({arreglo}, idx + 1)")))


def interp_fijo(arreglo, T):
    j = int(np.clip(np.searchsorted(PERIODOS, T) - 1, 0, len(PERIODOS) - 2))
    wj = float(np.clip((np.log(T) - np.log(PERIODOS[j])) / (np.log(PERIODOS[j + 1]) - np.log(PERIODOS[j])), 0, 1))
    return F.exp((1 - wj) * F.log(F.expr(f"element_at({arreglo}, {j + 1})")) + wj * F.log(F.expr(f"element_at({arreglo}, {j + 2})")))


def sa_sitio(T):
    return interp_fijo("sa_roca", T) * interp_fijo("F_arr", T)


ed = (spark.table("gold_edificios")
      .select("id", "ubigeo", "distrito", "provincia", "h3_id", "h3_r8", "pisos", "Tb", "TP", "TL", "S", "Tb_TP", "vs30", "tipo_suelo")
      .where("vs30 IS NOT NULL")
      .withColumn("vs30_bin", F.least(F.lit(float(VS30_GRID.max())),
                                      F.greatest(F.lit(float(VS30_GRID.min())), F.round(F.col("vs30") / 10) * 10))))
print(f"Champion de amplificación usado en 07: v{param['version_amplificacion']} ({param['algoritmo_amplificacion']})")

## 5. Acelerograma sintético con los parámetros calibrados en 07

In [ ]:
if "acelerograma_sintetico" not in param:
    raise RuntimeError("Falta la calibración del acelerograma sintético: ejecute antes 07_demanda_sismica")
SINT = param["acelerograma_sintetico"]
RHO_G_CM3, BETA_KM_S, FS_SIM = float(SINT["rho_g_cm3"]), float(SINT["beta_km_s"]), int(SINT["fs_hz"])
PERIODOS_CAL, C_SINT = np.array(SINT["periodos_cal"], float), np.array(SINT["correccion"], float)
PARAM_SINT = (float(SINT["delta_sigma"]), float(SINT["kappa"]), float(SINT["q0"]))


def espectro_fourier(fr, m, r, ds, ka, q0):
    m0 = 10 ** (1.5 * m + 16.05)
    fc = 4.9e6 * BETA_KM_S * (ds / m0) ** (1 / 3)
    re_km = np.sqrt(r ** 2 + (10 ** (-0.405 + 0.235 * m)) ** 2)
    g = 1 / re_km if re_km <= 70 else (1 / 70 if re_km <= 130 else np.sqrt(130 / re_km) / 70)
    q = np.maximum(q0 * fr ** 0.5, 50.0)
    c = 0.55 * 0.707 * 2.0 / (4 * np.pi * RHO_G_CM3 * BETA_KM_S ** 3) * 1e-20
    return c * m0 * (2 * np.pi * fr) ** 2 / (1 + (fr / fc) ** 2) * g * np.exp(-np.pi * fr * re_km / (q * BETA_KM_S)) * np.exp(-np.pi * ka * fr), fc, re_km


def sintetico_roca(m, r, semilla, ds, ka, q0, correccion=None):
    rng = np.random.default_rng(semilla)
    _, fc, re_km = espectro_fourier(np.array([1.0]), m, r, ds, ka, q0)
    tgm = 1 / fc + 0.05 * re_km
    n = int(2 ** np.ceil(np.log2((2.5 * tgm + 20) * FS_SIM)))
    t = np.arange(n) / FS_SIM
    eps, tn = 0.2, 2 * tgm
    b = -eps * np.log(0.05) / (1 + eps * (np.log(eps) - 1))
    ventana = (np.e / (eps * tn)) ** b * np.maximum(t, 1e-9) ** b * np.exp(-(b / (eps * tn)) * t)
    x = np.fft.rfft(rng.standard_normal(n) * ventana)
    x /= np.sqrt(np.mean(np.abs(x) ** 2))
    fr = np.fft.rfftfreq(n, 1 / FS_SIM)
    fr[0] = fr[1]
    a, _, _ = espectro_fourier(fr, m, r, ds, ka, q0)
    if correccion is not None:
        a = a * np.exp(np.interp(np.log(1 / fr), np.log(PERIODOS_CAL), correccion))
    return np.fft.irfft(a * x * FS_SIM, n)


print(f"Acelerograma sintético calibrado en 07: Δσ = {PARAM_SINT[0]:g} bar, κ = {PARAM_SINT[1]:g} s, Q0 = {PARAM_SINT[2]:g}")

## 6. Escenarios

In [ ]:
ev_real = REG_LIMA.where(F.col("evento") == EVENTO_EFECTO).select("ev_lat", "ev_lon", "ev_prof_km").first()
SISMOS_TIEMPO_REAL = [
    {"sismo": "callao_2025_m61", "magnitud": 6.1, "lat": ev_real.ev_lat, "lon": ev_real.ev_lon, "prof_km": ev_real.ev_prof_km, "semilla": SEED, "escenario_real": f"{EVENTO_EFECTO}_registrado"},
    {"sismo": "callao_m75", "magnitud": 7.5, "lat": ev_real.ev_lat, "lon": ev_real.ev_lon, "prof_km": ev_real.ev_prof_km, "semilla": SEED + 1000},
    {"sismo": "callao_m80", "magnitud": 8.0, "lat": ev_real.ev_lat, "lon": ev_real.ev_lon, "prof_km": ev_real.ev_prof_km, "semilla": SEED + 2000},
]
VP_KM_S, VS_KM_S = 7.0, 4.0
RAZON_P_S = 0.15
RUIDO_G = 1e-4
DURACION_MAX_S = 90
LOTE_S = 10
STA_S, LTA_S, UMBRAL_STA_LTA = 0.5, 5.0, 4.0
VENTANA_P_S, PASO_P_S, UMBRAL_P_MODELO = 4.0, 0.5, 0.5
N_VENTANA_P = int(VENTANA_P_S * FS_SIM)
RAZON_V_H, RAZON_PZ_PH = 0.67, 2.0
UMBRAL_SACUDIDA_G = 0.05
BASE_US = pd.Timestamp("2030-01-01", tz="America/Lima").value // 1000
VOLUMEN_CHECKPOINT = f"{volumen}/checkpoints"
vs30_distrito = (spark.table("gold_edificios").groupBy("ubigeo").agg(F.expr("percentile_approx(vs30, 0.5)").alias("vs30_mediana")).toPandas())
BASE_DISTRITOS = (dist[dist.en_estudio.astype(bool)][["ubigeo", "distrito", "centroide_lon", "centroide_lat"]].merge(vs30_distrito, on="ubigeo").sort_values("ubigeo").reset_index(drop=True))


def llegada_distrito(s, c):
    r = float(np.sqrt(((c.centroide_lon - s["lon"]) * 111.32 * np.cos(np.radians((c.centroide_lat + s["lat"]) / 2))) ** 2
                      + ((c.centroide_lat - s["lat"]) * 110.57) ** 2 + s["prof_km"] ** 2))
    return {"sismo": s["sismo"], "ubigeo": c.ubigeo, "distrito": c.distrito, "dist_hipo_km": r, "t_llegada_p_s": r / VP_KM_S, "t_llegada_s_s": r / VS_KM_S, "vs30_mediana": float(c.vs30_mediana)}


def serie_distrito(s, c, j):
    rng = np.random.default_rng(s["semilla"] + 100 * j + 50)
    llegada = llegada_distrito(s, c)
    r, t_p, t_s = llegada["dist_hipo_km"], llegada["t_llegada_p_s"], llegada["t_llegada_s_s"]
    roca = [sintetico_roca(s["magnitud"], r, s["semilla"] + 100 * j + i, *PARAM_SINT, C_SINT) for i in (0, 1, 2)]
    n = min(len(x) for x in roca)
    roca = [x[:n] for x in roca]
    pga_r = float(np.sqrt(np.abs(roca[0]).max() * np.abs(roca[1]).max()) / G_CM_S2)
    fr = np.fft.rfftfreq(n, 1 / FS_SIM)
    fr[0] = fr[1]
    transferencia = np.exp(ln_F(1 / fr, c.vs30_mediana, pga_r))
    suelo = [np.fft.irfft(np.fft.rfft(x) * transferencia, n) for x in roca[:2]]
    roca_g = [filtrar_cms2(x, FS_SIM) / G_CM_S2 for x in roca]
    suelo_g = [filtrar_cms2(x, FS_SIM) / G_CM_S2 for x in suelo]
    arias = intensidad_arias(suelo_g[0], FS_SIM)
    i_fin = int(min(np.searchsorted(arias, 0.99 * arias[-1]) + 5 * FS_SIM, DURACION_MAX_S * FS_SIM, n))
    i_p, i_s = int(t_p * FS_SIM), int(t_s * FS_SIM)
    columnas = {}
    for nombre, base in (("ew_g", suelo_g[0]), ("ns_g", suelo_g[1]), ("ew_roca_g", roca_g[0]), ("ns_roca_g", roca_g[1])):
        x = rng.normal(0.0, RUIDO_G, i_s + i_fin)
        fase_p = rng.standard_normal(i_s - i_p)
        fase_p = sosfiltfilt(butter(4, (1.0, 25.0), btype="bandpass", fs=FS_SIM, output="sos"), fase_p) * tukey(len(fase_p), 0.2)
        x[i_p:i_s] += fase_p / max(fase_p.std(), 1e-12) * RAZON_P_S * base[:i_fin].std()
        x[i_s:i_s + i_fin] += base[:i_fin]
        columnas[nombre] = x
    x = rng.normal(0.0, RUIDO_G, i_s + i_fin)
    fase_p = sosfiltfilt(butter(4, (1.0, 25.0), btype="bandpass", fs=FS_SIM, output="sos"), rng.standard_normal(i_s - i_p)) * tukey(i_s - i_p, 0.2)
    x[i_p:i_s] += fase_p / max(fase_p.std(), 1e-12) * RAZON_P_S * RAZON_PZ_PH * suelo_g[0][:i_fin].std()
    x[i_s:i_s + i_fin] += RAZON_V_H * roca_g[2][:i_fin]
    columnas["z_g"] = x
    return pd.DataFrame({"sismo": s["sismo"], "ubigeo": c.ubigeo, "distrito": c.distrito, "muestra": np.arange(i_s + i_fin), **columnas})

In [ ]:
RASGOS_VENTANA = [f"{r}_{c}" for c in "enz" for r in ("log_energia", "log_max", "curtosis", "log_ratio_mitades")] + ["frec_dominante_z", "log_z_sobre_h"]
ESQ_RASGOS_P = "sismo string, ubigeo string, primera_muestra long, " + ", ".join(f"{c} double" for c in RASGOS_VENTANA)


def rasgos_ventana(x):
    x = x - x.mean(0)
    mitad = len(x) // 2
    e1 = (x[:mitad] ** 2).mean(0) + 1e-12
    e2 = (x[mitad:] ** 2).mean(0) + 1e-12
    r = {}
    for i, c in enumerate("enz"):
        r[f"log_energia_{c}"] = float(np.log10((x[:, i] ** 2).mean() + 1e-12))
        r[f"log_max_{c}"] = float(np.log10(np.abs(x[:, i]).max() + 1e-12))
        r[f"curtosis_{c}"] = float(kurtosis(x[:, i]))
        r[f"log_ratio_mitades_{c}"] = float(np.log10(e2[i] / e1[i]))
    espectro = np.abs(np.fft.rfft(x[:, 2]))
    r["frec_dominante_z"] = float(np.fft.rfftfreq(len(x), 1 / FS_SIM)[espectro.argmax()])
    r["log_z_sobre_h"] = float(np.log10(((x[:, 2] ** 2).mean() + 1e-12) / ((x[:, :2] ** 2).mean() + 1e-12)))
    return r


ruido_stead = (spark.table("gold_onda_p_ventanas").where(F.col("tipo") == "ruido_previo")
               .agg(F.expr("percentile_approx(log_energia_e, 0.5)")).first()[0])
ESCALA_STEAD = float(np.sqrt(10 ** ruido_stead) / RUIDO_G)
print(f"Champion de onda P: v{meta_onda['version']} ({meta_onda['algoritmo']}), variables {meta_onda['cols_num']}")
print(f"Escala del sintético a STEAD: {ESCALA_STEAD:.3g} (el ruido queda con la energía mediana del ruido previo de STEAD)")

## 7. Simulación en streaming: onda P, amplificación y edificios

In [ ]:
ESQ_SERIE = "sismo string, ubigeo string, distrito string, muestra long, ew_g double, ns_g double, ew_roca_g double, ns_roca_g double, z_g double"
ESQ_ONDAS_DISTRITO = ("ubigeo string, sa_roca array<double>, sa_suelo array<double>, F_lineal array<double>, F_no_lineal array<double>, "
                      "pga_roca_g double, pga_suelo_g double, pgv_cms double, pgv_suelo_cms double, arias_ms double, d595_s double")
ESQ_SEGUIMIENTO = ("sismo string, lote long, senal_s double, proceso_s double, reloj_s double, distritos_con_p long, "
                   "distritos_con_p_modelo long, pico_g double")
VS30_DISTRITO = dict(zip(BASE_DISTRITOS.ubigeo, BASE_DISTRITOS.vs30_mediana.astype(float)))
COLS_DANO_SIM = ["id", "ubigeo", "distrito", "tipologia", "pisos", "vs30", "PGA_g", "PGV_cms", "Sa03_g", "Sa06_g", "Sa10_g", "Sa_Tb", "IM_g",
                 "indice_dano"] + [f"P_ge_{e}" for e in ESTADOS]


def guardar_por_sismo(sdf, tabla, sismo):
    if spark.catalog.tableExists(tabla):
        (sdf.write.format("delta").mode("overwrite").option("mergeSchema", "true")
         .option("replaceWhere", f"sismo = '{sismo}'").saveAsTable(tabla))
    else:
        sdf.write.format("delta").mode("overwrite").partitionBy("sismo").saveAsTable(tabla)


def ondas_distrito(g):
    g = g.sort_values("muestra")
    e_r, n_r, e_s, n_s = [g[c].to_numpy(float) for c in ("ew_roca_g", "ns_roca_g", "ew_g", "ns_g")]
    sa_r = np.sqrt(espectro_sa(e_r, 1 / FS_SIM) * espectro_sa(n_r, 1 / FS_SIM))
    sa_s = np.sqrt(espectro_sa(e_s, 1 / FS_SIM) * espectro_sa(n_s, 1 / FS_SIM))
    pga_r = float(np.sqrt(np.abs(e_r).max() * np.abs(n_r).max()))
    vs30 = VS30_DISTRITO[g["ubigeo"].iloc[0]]
    pgv_s = float(np.sqrt(np.abs(cumulative_trapezoid(e_s * G_CM_S2, dx=1 / FS_SIM)).max() * np.abs(cumulative_trapezoid(n_s * G_CM_S2, dx=1 / FS_SIM)).max()))
    ia_e, ia_n = intensidad_arias(e_s, FS_SIM), intensidad_arias(n_s, FS_SIM)
    return pd.DataFrame([{"ubigeo": g["ubigeo"].iloc[0], "sa_roca": sa_r.tolist(), "sa_suelo": sa_s.tolist(),
                          "F_lineal": np.exp(ln_amplificacion(PERIODOS, vs30)).tolist(), "F_no_lineal": np.exp(ln_F(PERIODOS, vs30, pga_r)).tolist(),
                          "pga_roca_g": pga_r, "pga_suelo_g": float(np.sqrt(np.abs(e_s).max() * np.abs(n_s).max())), "pgv_cms": pgv_s,
                          "pgv_suelo_cms": pgv_s, "arias_ms": float(np.sqrt(ia_e[-1] * ia_n[-1])), "d595_s": float(np.sqrt(duracion_d595(ia_e, FS_SIM) * duracion_d595(ia_n, FS_SIM)))}])


def rasgos_lote(lotes):
    for pdf in lotes:
        filas = [{"sismo": r.sismo, "ubigeo": r.ubigeo, "primera_muestra": int(r.primera_muestra),
                  **rasgos_ventana(np.column_stack([r.e, r.n, r.z]).astype(float) * ESCALA_STEAD)}
                 for r in pdf.itertuples() if len(r.e) == N_VENTANA_P]
        yield pd.DataFrame(filas, columns=["sismo", "ubigeo", "primera_muestra"] + RASGOS_VENTANA)


def correr_sismo(s):
    topico, ventanas, ventanas_p, puntajes_p = f"{TR}.sismo_stream_acelerograma", f"{TR}.sismo_stream_ventanas", f"{TR}.sismo_stream_ventanas_p", f"{TR}.sismo_tmp_puntajes_p"
    checkpoint = f"{VOLUMEN_CHECKPOINT}/{s['sismo']}_{uuid.uuid4().hex[:8]}"
    for t in (topico, ventanas, ventanas_p, puntajes_p):
        spark.sql(f"DROP TABLE IF EXISTS {t}")

    def series_lote(lotes):
        for pdf in lotes:
            for j in pdf["j"]:
                yield serie_distrito(s, BASE_DISTRITOS.iloc[int(j)], int(j))

    n_distritos = len(BASE_DISTRITOS)
    escribir(spark.range(n_distritos).toDF("j").repartition(n_distritos).mapInPandas(series_lote, ESQ_SERIE), f"{TR}.sismo_tmp_serie", contar=False)
    serie = spark.table(f"{TR}.sismo_tmp_serie")
    llegadas = a_spark(pd.DataFrame([llegada_distrito(s, c) for _, c in BASE_DISTRITOS.iterrows()]))
    duracion = (serie.agg(F.max("muestra")).first()[0] + 1) / FS_SIM

    def consumidor():
        (spark.readStream.table(topico)
         .withWatermark("ts", "1 second")
         .groupBy("sismo", "ubigeo", "distrito", F.window("ts", f"{int(STA_S * 1000)} milliseconds"))
         .agg(F.sum(F.col("ew_g") ** 2 + F.col("ns_g") ** 2).alias("energia"), F.count("*").alias("muestras"), F.max("muestra").alias("ultima_muestra"),
              F.max(F.greatest(F.abs("ew_g"), F.abs("ns_g"))).alias("pico_g"))
         .select("sismo", "ubigeo", "distrito", F.col("window.start").alias("inicio"), "energia", "muestras", "ultima_muestra", "pico_g")
         .writeStream.outputMode("append").option("checkpointLocation", checkpoint)
         .trigger(availableNow=True).toTable(ventanas)
         .awaitTermination())

    def consumidor_p():
        (spark.readStream.table(topico)
         .withWatermark("ts", "1 second")
         .groupBy("sismo", "ubigeo", F.window("ts", f"{int(VENTANA_P_S * 1000)} milliseconds", f"{int(PASO_P_S * 1000)} milliseconds"))
         .agg(F.array_sort(F.collect_list(F.struct("muestra", "ew_g", "ns_g", "z_g"))).alias("x"))
         .select("sismo", "ubigeo", F.expr("x[0].muestra").alias("primera_muestra"), F.expr("transform(x, v -> v.ew_g)").alias("e"),
                 F.expr("transform(x, v -> v.ns_g)").alias("n"), F.expr("transform(x, v -> v.z_g)").alias("z"))
         .writeStream.outputMode("append").option("checkpointLocation", f"{checkpoint}_p")
         .trigger(availableNow=True).toTable(ventanas_p)
         .awaitTermination())

    def detectar():
        n_lta = int(LTA_S / STA_S)
        previas = Window.partitionBy("ubigeo").orderBy("inicio").rowsBetween(-n_lta, -1)
        return (spark.table(ventanas)
                .withColumn("sta", F.col("energia") / F.col("muestras"))
                .withColumn("lta", F.avg("sta").over(previas)).withColumn("n_lta", F.count("sta").over(previas))
                .withColumn("sta_lta", F.col("sta") / F.col("lta"))
                .where((F.col("n_lta") == n_lta) & (F.col("sta_lta") > UMBRAL_STA_LTA))
                .withColumn("_orden", F.row_number().over(Window.partitionBy("ubigeo").orderBy("inicio"))).where("_orden = 1")
                .select("ubigeo", "ultima_muestra", "sta_lta"))

    def detectar_modelo():
        return (spark.table(puntajes_p).where(F.col("p_onda_p") >= UMBRAL_P_MODELO)
                .groupBy("ubigeo").agg(F.min("primera_muestra").alias("primera_muestra")))

    seguimiento, n_detectados, n_detectados_modelo, ultima_ventana = [], 0, 0, -1
    inicio_real = time.perf_counter()
    for k in range(int(np.ceil(duracion / LOTE_S))):
        t0 = time.perf_counter()
        lote = (serie.where((F.col("muestra") >= k * LOTE_S * FS_SIM) & (F.col("muestra") < (k + 1) * LOTE_S * FS_SIM))
                .withColumn("ts", F.timestamp_micros(F.lit(BASE_US) + F.col("muestra") * (1_000_000 // FS_SIM))))
        lote.write.mode("append").saveAsTable(topico)
        consumidor()
        consumidor_p()
        if n_detectados < n_distritos:
            n_detectados = detectar().count()
        maximo = spark.table(ventanas_p).agg(F.max("primera_muestra")).first()[0]
        if maximo is not None and maximo > ultima_ventana:
            (modelo_onda.transform(spark.table(ventanas_p).where(F.col("primera_muestra") > ultima_ventana).mapInPandas(rasgos_lote, ESQ_RASGOS_P))
             .select("sismo", "ubigeo", "primera_muestra", *RASGOS_VENTANA, vector_to_array("probability")[1].alias("p_onda_p"))
             .write.mode("append").saveAsTable(puntajes_p))
            ultima_ventana = maximo
            if n_detectados_modelo < n_distritos:
                n_detectados_modelo = detectar_modelo().count()
        pico = spark.table(ventanas).agg(F.max("pico_g")).first()[0]
        senal = min((k + 1) * LOTE_S, duracion)
        proceso = time.perf_counter() - t0
        seguimiento.append({"sismo": s["sismo"], "lote": k, "senal_s": float(senal), "proceso_s": round(proceso, 1), "reloj_s": round(time.perf_counter() - inicio_real, 1),
                            "distritos_con_p": n_detectados, "distritos_con_p_modelo": n_detectados_modelo, "pico_g": None if pico is None else float(pico)})
        print(f"  {s['sismo']} | señal {senal:5.1f} s | proceso {proceso:4.1f} s | P en {n_detectados}/{n_distritos} (STA/LTA) "
              f"y {n_detectados_modelo}/{n_distritos} (champion) | pico {0.0 if pico is None else pico:.3f} g")
        time.sleep(max(0.0, LOTE_S - (time.perf_counter() - t0)))

    guardar_por_sismo(spark.table(topico), f"{TR}.sismo_acelerogramas", s["sismo"])
    guardar_por_sismo(a_spark(pd.DataFrame(seguimiento), ESQ_SEGUIMIENTO), f"{TR}.sismo_seguimiento", s["sismo"])
    llegada_p = llegadas.select("ubigeo", "t_llegada_p_s")
    antes_de_p = (F.col("primera_muestra") + N_VENTANA_P) / FS_SIM <= F.col("t_llegada_p_s")
    guardar_por_sismo(spark.table(puntajes_p).join(llegada_p, "ubigeo")
                      .withColumn("inicio_s", F.col("primera_muestra") / FS_SIM).withColumn("fin_s", (F.col("primera_muestra") + N_VENTANA_P) / FS_SIM)
                      .withColumn("antes_de_p", antes_de_p).drop("t_llegada_p_s"), f"{TR}.sismo_onda_p_ventanas", s["sismo"])
    alarma = F.col("p_onda_p") >= UMBRAL_P_MODELO
    modelo_p = (spark.table(puntajes_p).join(llegada_p, "ubigeo").groupBy("ubigeo")
                .agg(F.min(F.when(alarma, F.col("primera_muestra"))).alias("m_alarma"),
                     F.sum((alarma & antes_de_p).cast("int")).alias("falsas_alarmas_modelo"),
                     F.sum(antes_de_p.cast("int")).alias("ventanas_antes_de_p"),
                     F.max(F.when(antes_de_p, F.col("p_onda_p"))).alias("p_max_antes_de_p"))
                .withColumn("t_deteccion_p_modelo_s", (F.col("m_alarma") + N_VENTANA_P) / FS_SIM).drop("m_alarma"))
    tiempos = (spark.table(topico).withColumn("abs_max", F.greatest(F.abs("ew_g"), F.abs("ns_g"))).groupBy("ubigeo")
               .agg(F.max("abs_max").alias("pico_suelo_g"), F.max_by("muestra", "abs_max").alias("m_pico"),
                    F.min(F.when(F.col("abs_max") >= UMBRAL_SACUDIDA_G, F.col("muestra"))).alias("m_sacudida")))
    escribir(spark.table(topico).select("ubigeo", "muestra", "ew_g", "ns_g", "ew_roca_g", "ns_roca_g").groupBy("ubigeo").applyInPandas(ondas_distrito, ESQ_ONDAS_DISTRITO),
             f"{TR}.sismo_tmp_ondas", contar=False)
    ondas = spark.table(f"{TR}.sismo_tmp_ondas")
    guardar_por_sismo(ondas.join(llegadas.select("ubigeo", "distrito", "vs30_mediana"), "ubigeo")
                      .withColumn("sismo", F.lit(s["sismo"])).withColumn("magnitud", F.lit(float(s["magnitud"]))), f"{TR}.sismo_amplificacion_distritos", s["sismo"])
    roca_s = ondas.select("ubigeo", "sa_roca", "pga_roca_g", "pgv_cms")
    dem = (ed.where(F.col("ubigeo").isin(BASE_DISTRITOS.ubigeo.tolist()))
           .join(F.broadcast(tb_tab), "Tb").join(F.broadcast(amp_arr), "vs30_bin")
           .join(F.broadcast(sitio_arr), "h3_r8", "left").withColumn("F_arr", F_SITIO)
           .join(F.broadcast(roca_s), "ubigeo").join(spark.table("gold_edificios").select("id", "area_m2"), "id")
           .withColumn("pga_r_nl", F.col("pga_roca_g")).withColumn("F_arr", F_NO_LINEAL)
           .withColumn("Sa_roca_Tb", interp_log("sa_roca")).withColumn("F_Tb", interp_log("F_arr"))
           .withColumn("Sa_Tb", F.col("Sa_roca_Tb") * F.col("F_Tb"))
           .withColumn("PGA_g", F.col("pga_roca_g") * F.expr("element_at(F_arr, 1)"))
           .withColumn("Sa03_g", sa_sitio(0.3)).withColumn("Sa06_g", sa_sitio(0.6)).withColumn("Sa10_g", sa_sitio(1.0)))
    guardar_por_sismo(dano_edificios(dem).select(F.lit(s["sismo"]).alias("sismo"), *COLS_DANO_SIM), f"{TR}.sismo_dano_edificios", s["sismo"])
    detecciones = detectar().select("ubigeo", ((F.col("ultima_muestra") + 1) / FS_SIM).alias("t_deteccion_p_s"))
    reloj_p = next((f["reloj_s"] for f in seguimiento if f["distritos_con_p"] > 0), None)
    reloj_p_modelo = next((f["reloj_s"] for f in seguimiento if f["distritos_con_p_modelo"] > 0), None)
    resultado = (llegadas.join(tiempos, "ubigeo").join(ondas.select("ubigeo", "pgv_suelo_cms", "arias_ms", "d595_s"), "ubigeo")
                 .join(detecciones, "ubigeo", "left").join(modelo_p, "ubigeo", "left")
                 .join(efecto_por_distrito(spark.table(f"{TR}.sismo_dano_edificios").where(F.col("sismo") == s["sismo"]), ["ubigeo", "distrito"]).drop("distrito"), "ubigeo")
                 .withColumn("t_pico_s", F.col("m_pico") / FS_SIM).withColumn("t_sacudida_s", F.col("m_sacudida") / FS_SIM)
                 .withColumn("aviso_teorico_s", F.col("t_llegada_s_s") - F.col("t_deteccion_p_s"))
                 .withColumn("aviso_modelo_s", F.col("t_llegada_s_s") - F.col("t_deteccion_p_modelo_s"))
                 .withColumn("proceso_medio_por_lote_s", F.lit(float(np.mean([f["proceso_s"] for f in seguimiento]))))
                 .withColumn("reloj_hasta_p_s", F.lit(reloj_p).cast("double"))
                 .withColumn("reloj_hasta_p_modelo_s", F.lit(reloj_p_modelo).cast("double"))
                 .withColumn("magnitud", F.lit(float(s["magnitud"]))).drop("m_pico", "m_sacudida"))
    guardar_por_sismo(resultado, f"{TR}.sismo_efecto_distritos", s["sismo"])
    for t in (topico, ventanas, ventanas_p, puntajes_p, f"{TR}.sismo_tmp_serie", f"{TR}.sismo_tmp_ondas"):
        spark.sql(f"DROP TABLE IF EXISTS {t}")


for s in SISMOS_TIEMPO_REAL:
    correr_sismo(s)
    display(spark.table(f"{TR}.sismo_efecto_distritos").where(F.col("sismo") == s["sismo"])
            .select("distrito", "dist_hipo_km", "t_llegada_p_s", "t_deteccion_p_s", "t_deteccion_p_modelo_s", "falsas_alarmas_modelo", "t_llegada_s_s",
                    "t_sacudida_s", "pico_suelo_g", "PGA_mediana_g", "P_ge_severo_medio", "indice_dano_medio", "pct_fuera_rango_PGA_g"))

## 8. Resultados de cada parte

In [ ]:
efecto = spark.table(f"{TR}.sismo_efecto_distritos").toPandas().sort_values(["magnitud", "ubigeo"])
efecto["retraso_sta_lta_s"] = efecto.t_deteccion_p_s - efecto.t_llegada_p_s
efecto["retraso_modelo_s"] = efecto.t_deteccion_p_modelo_s - efecto.t_llegada_p_s
display(efecto[["sismo", "distrito", "t_llegada_p_s", "t_deteccion_p_s", "retraso_sta_lta_s", "t_deteccion_p_modelo_s", "retraso_modelo_s", "falsas_alarmas_modelo",
                "p_max_antes_de_p", "t_llegada_s_s", "aviso_teorico_s", "aviso_modelo_s"]].round(2))
for nombre, columna in (("sta_lta", "t_deteccion_p_s"), ("champion_onda_p", "t_deteccion_p_modelo_s")):
    retraso, aviso = efecto[columna] - efecto.t_llegada_p_s, efecto.t_llegada_s_s - efecto[columna]
    extra = ({"falsas_alarmas": float(efecto.falsas_alarmas_modelo.sum()),
              "tasa_falsas_alarmas": float(efecto.falsas_alarmas_modelo.sum() / max(efecto.ventanas_antes_de_p.sum(), 1))} if nombre == "champion_onda_p" else {})
    registrar_modelo(f"deteccion_p_{nombre}", {"version_champion": meta_onda["version"], "umbral": UMBRAL_P_MODELO if nombre == "champion_onda_p" else UMBRAL_STA_LTA},
                     {"distritos_detectados": float(retraso.notna().sum()), "retraso_medio_s": float(retraso.mean()), "aviso_medio_s": float(aviso.mean()), **extra})

s_graf, d_graf = SISMOS_TIEMPO_REAL[-1]["sismo"], BASE_DISTRITOS.iloc[0]
filtro = (F.col("sismo") == s_graf) & (F.col("ubigeo") == d_graf.ubigeo)
serie_g = spark.table(f"{TR}.sismo_acelerogramas").where(filtro).select("muestra", "ew_g", "z_g").orderBy("muestra").toPandas()
p_g = spark.table(f"{TR}.sismo_onda_p_ventanas").where(filtro).select("fin_s", "p_onda_p").orderBy("fin_s").toPandas()
fila = efecto[(efecto.sismo == s_graf) & (efecto.ubigeo == d_graf.ubigeo)].iloc[0]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
a1.plot(serie_g.muestra / FS_SIM, serie_g.ew_g, lw=.4, label="EW (suelo)")
a1.plot(serie_g.muestra / FS_SIM, serie_g.z_g, lw=.4, label="Z")
a2.plot(p_g.fin_s, p_g.p_onda_p, ".-", lw=.8)
a2.axhline(UMBRAL_P_MODELO, color="k", ls=":", lw=1)
for eje in (a1, a2):
    for t, color, etiqueta in ((fila.t_llegada_p_s, "g", "llega la P"), (fila.t_deteccion_p_s, "b", "STA/LTA"),
                               (fila.t_deteccion_p_modelo_s, "r", "champion"), (fila.t_llegada_s_s, "m", "llega la S")):
        if pd.notna(t):
            eje.axvline(t, color=color, ls="--", lw=1, label=etiqueta if eje is a1 else None)
a1.set(ylabel="g", title=f"{s_graf} en {d_graf.distrito}: detección de la onda P", xlim=(0, fila.t_llegada_s_s + 20))
a1.legend(fontsize=7, ncol=3)
a2.set(xlabel="Segundos desde el origen (fin de cada ventana de 4 s)", ylabel="P(onda P)", ylim=(0, 1.02))
plt.tight_layout(); plt.show()

In [ ]:
amp_sim = spark.table(f"{TR}.sismo_amplificacion_distritos").toPandas().sort_values(["magnitud", "distrito"])
for T in (0.2, 1.0):
    j = int(np.abs(PERIODOS - T).argmin())
    amp_sim[f"F_champion_T{T}"] = [v[j] for v in amp_sim.F_lineal]
    amp_sim[f"F_no_lineal_T{T}"] = [v[j] for v in amp_sim.F_no_lineal]
    amp_sim[f"razon_espectral_T{T}"] = [s_[j] / r_[j] for s_, r_ in zip(amp_sim.sa_suelo, amp_sim.sa_roca)]
display(amp_sim[["sismo", "distrito", "vs30_mediana", "pga_roca_g", "pga_suelo_g"] + [c for c in amp_sim.columns if c[-4:] in ("T0.2", "T1.0")]].round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
for k, f in enumerate(amp_sim[amp_sim.sismo == s_graf].itertuples()):
    ax.loglog(PERIODOS, f.sa_roca, "--", color=f"C{k}", lw=1)
    ax.loglog(PERIODOS, f.sa_suelo, "-", color=f"C{k}", lw=1.5, label=f"{f.distrito} (Vs30 {f.vs30_mediana:.0f} m/s)")
ax.set(xlabel="Periodo (s)", ylabel="Sa (g)", title=f"{s_graf}: espectro en roca (discontinua) y en suelo (continua)")
ax.legend(fontsize=7); ax.grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()

In [ ]:
display(spark.table(f"{TR}.sismo_efecto_distritos").orderBy("magnitud", "distrito")
        .select("sismo", "distrito", "edificios", "PGA_mediana_g", "Sa_Tb_mediana_g", "P_ge_severo_medio", "P_colapso_medio",
                "edificios_severo_esperados", "indice_dano_medio", "pct_indice_mayor_05", "pct_fuera_rango_PGA_g"))
display(spark.table(f"{TR}.sismo_dano_edificios").groupBy("sismo", "tipologia")
        .agg(F.count("*").alias("edificios"), F.round(F.avg("P_ge_severo"), 3).alias("P_ge_severo_medio"),
             F.round(F.avg("P_ge_colapso"), 3).alias("P_colapso_medio"), F.round(F.avg("indice_dano"), 3).alias("indice_dano_medio"))
        .orderBy("sismo", "tipologia"))
ef = spark.table(f"{TR}.sismo_efecto_distritos").select("sismo", "distrito", "P_ge_severo_medio").toPandas()
ax = (ef.pivot_table(index="distrito", columns="sismo", values="P_ge_severo_medio")[[s["sismo"] for s in SISMOS_TIEMPO_REAL]]
      .plot.bar(figsize=(9, 4), rot=0))
ax.set(ylabel="P(daño ≥ severo) media (GEM)", title="Daño en los edificios por sismo simulado")
plt.tight_layout(); plt.show()

## 9. Validación con el sismo de Callao 2025

In [ ]:
s_val = SISMOS_TIEMPO_REAL[0]
sim = spark.table(f"{TR}.sismo_efecto_distritos").where(F.col("sismo") == s_val["sismo"]).select("distrito", F.col("PGA_mediana_g").alias("PGA_sintetico"), F.col("Sa_Tb_mediana_g").alias("Sa_Tb_sintetico"))
real = (spark.table("riesgo_demanda").where(F.col("escenario") == s_val["escenario_real"]).groupBy("distrito")
        .agg(F.expr("percentile_approx(PGA_g, 0.5)").alias("PGA_registrado"), F.expr("percentile_approx(Sa_Tb, 0.5)").alias("Sa_Tb_registrado")))
escribir(sim.join(real, "distrito").withColumn("razon_PGA", F.col("PGA_sintetico") / F.col("PGA_registrado")).withColumn("razon_Sa_Tb", F.col("Sa_Tb_sintetico") / F.col("Sa_Tb_registrado")), f"{TR}.sismo_validacion")
comparacion = spark.table(f"{TR}.sismo_validacion")
display(comparacion)
errores = comparacion.agg(F.sqrt(F.avg(F.log("razon_PGA") ** 2)).alias("pga"), F.sqrt(F.avg(F.log("razon_Sa_Tb") ** 2)).alias("sa")).first()
registrar_modelo("sismo_tiempo_real", {"lote_s": LOTE_S, "sta_s": STA_S, "lta_s": LTA_S, "umbral": UMBRAL_STA_LTA, "razon_p_s": RAZON_P_S, "validacion": s_val["escenario_real"], "champion_onda_p": meta_onda["version"]},
                 {"rmse_ln_pga_distritos": errores.pga, "rmse_ln_sa_tb_distritos": errores.sa})

acel = spark.table(f"{TR}.sismo_acelerogramas").where(F.col("distrito") == BASE_DISTRITOS.distrito.iloc[0]).select("sismo", "muestra", "ew_g").orderBy("sismo", "muestra").toPandas()
fig, ejes = plt.subplots(len(SISMOS_TIEMPO_REAL), 1, figsize=(11, 2.4 * len(SISMOS_TIEMPO_REAL)), sharex=True)
for eje, s in zip(np.atleast_1d(ejes), SISMOS_TIEMPO_REAL):
    g = acel[acel.sismo == s["sismo"]]
    eje.plot(g.muestra / FS_SIM, g.ew_g, lw=.4)
    eje.set(ylabel="g", title=f"{s['sismo']} (M{s['magnitud']}) en {BASE_DISTRITOS.distrito.iloc[0]}")
eje.set_xlabel("Segundos desde el origen")
plt.tight_layout(); plt.show()
display(spark.table(f"{TR}.sismo_efecto_distritos").orderBy("magnitud", F.desc("indice_dano_medio"))
        .select("sismo", "magnitud", "distrito", "t_llegada_p_s", "t_llegada_s_s", "t_sacudida_s", "t_pico_s", "reloj_hasta_p_s", "proceso_medio_por_lote_s", "PGA_mediana_g", "indice_dano_medio", "pct_indice_mayor_05", "pct_fuera_rango_PGA_g", "t_deteccion_p_modelo_s", "aviso_modelo_s", "P_ge_severo_medio"))

## 10. Datos del simulador web

In [ ]:
def arboles_pipeline(ruta):
    etapa = {int(os.path.basename(p).split("_")[0]): p for p in glob.glob(f"{ruta}/stages/*")}
    leer_meta = lambda p: json.loads(open(sorted(glob.glob(f"{p}/metadata/part*"))[0]).read())
    clase = leer_meta(etapa[2])["class"]
    escala = pq.read_table(f"{etapa[1]}/data").to_pylist()[0]
    pesos = {r["_1"]: r["_3"] for r in pq.read_table(f"{etapa[2]}/treesMetadata").to_pylist()}
    nodos = {}
    for r in pq.read_table(f"{etapa[2]}/data").to_pylist():
        nodos.setdefault(r["treeID"], []).append(r["nodeData"])
    arboles = []
    for tid in sorted(nodos):
        ns = sorted(nodos[tid], key=lambda n: n["id"])
        hoja = [n["leftChild"] < 0 for n in ns]
        arboles.append({"f": [-1 if h else n["split"]["featureIndex"] for n, h in zip(ns, hoja)],
                        "t": [0.0 if h else n["split"]["leftCategoriesOrThreshold"][0] for n, h in zip(ns, hoja)],
                        "l": [n["leftChild"] for n in ns], "r": [n["rightChild"] for n in ns],
                        "v": [n["prediction"] if "GBT" in clase else (n["impurityStats"][1] / sum(n["impurityStats"]) if h else 0.0)
                              for n, h in zip(ns, hoja)]})
    return {"tipo": "gbt" if "GBT" in clase else "rf", "columnas": leer_meta(etapa[0])["paramMap"]["inputCols"],
            "media": escala["mean"]["values"], "desv": escala["std"]["values"], "pesos": [pesos[t] for t in sorted(nodos)], "arboles": arboles}


def filtro_sos(banda):
    sos = butter(4, banda, btype="bandpass", fs=FS_SIM, output="sos")
    ntaps = 2 * len(sos) + 1 - min(int((sos[:, 2] == 0).sum()), int((sos[:, 5] == 0).sum()))
    return {"sos": sos.tolist(), "zi": sosfilt_zi(sos).tolist(), "padlen": 3 * ntaps}


def columnas_json(df, decimales=None):
    decimales = decimales or {}
    return {c: [None if pd.isna(v) else (round(float(v), decimales[c]) if c in decimales else (v.item() if hasattr(v, "item") else v))
                for v in df[c]] for c in df.columns}

In [ ]:
SIM_WEB = f"{volumen}/web/simulador/simulador.json"
N_MUESTRA_RF = 120
ubigeos = BASE_DISTRITOS.ubigeo.tolist()
base_sim = (ed.where(F.col("ubigeo").isin(ubigeos)).join(F.broadcast(CENSO), "ubigeo", "left")
            .fillna(0.0, subset=["p_adobe", "p_quincha", "p_madera"]).withColumn("tipologia", tipologia_gem())
            .withColumn("pisos_gem", F.least(F.col("pisos"), tope_gem())))
grupos = base_sim.groupBy("ubigeo", "h3_r8", "vs30_bin", "tipologia", "pisos_gem").count().toPandas()
muestra = (base_sim.join(spark.table("gold_edificios").select("id", "area_m2").dropDuplicates(["id"]), "id")
           .withColumn("_k", F.row_number().over(Window.partitionBy("ubigeo").orderBy(F.xxhash64("id")))).where(F.col("_k") <= N_MUESTRA_RF)
           .select("ubigeo", "h3_r8", "vs30", "vs30_bin", "pisos", "Tb", "area_m2", "tipologia").toPandas())
hexes = (base_sim.select("h3_r8").where(F.col("h3_r8").isNotNull()).distinct().join(sitio_arr, "h3_r8", "left")
         .withColumn("borde", F.expr("h3_boundaryasgeojson(h3_r8)")).orderBy("h3_r8").toPandas())
contornos = dict(spark.table("silver_distritos").where(F.col("ubigeo").isin(ubigeos))
                 .select("ubigeo", F.expr("st_asgeojson(st_simplify(st_geomfromwkt(geometry_wkt), 0.0004))")).toPandas().itertuples(index=False, name=None))
i_dist, i_hex = {u: k for k, u in enumerate(ubigeos)}, {h: k for k, h in enumerate(hexes.h3_r8)}
tipologias = sorted(set(grupos.tipologia) | set(muestra.tipologia))
for df in (grupos, muestra):
    df["d"], df["h"] = df.ubigeo.map(i_dist), df.h3_r8.map(i_hex).fillna(-1).astype(int)
    df["v"], df["t"] = ((df.vs30_bin - VS30_GRID[0]) / 10).round().astype(int), df.tipologia.map({t: k for k, t in enumerate(tipologias)})

SISMO_VERIF = SISMOS_TIEMPO_REAL[-1]["sismo"]
MODELO_ONDA, MODELO_DANO = arboles_pipeline(meta_onda["ruta"]), arboles_pipeline(meta_dano["ruta"])
del_sismo = lambda t: spark.table(f"{TR}.{t}").where(F.col("sismo") == SISMO_VERIF)
unicos = spark.table("gold_edificios").groupBy("id").agg(F.count("*").alias("_n"), F.first("Tb").alias("Tb"), F.first("area_m2").alias("area_m2")).where("_n = 1")
ver_ventanas = del_sismo("sismo_onda_p_ventanas").orderBy(F.xxhash64("ubigeo", "primera_muestra")).limit(300).select(*MODELO_ONDA["columnas"], "p_onda_p").toPandas()
ver_edificios = (del_sismo("sismo_dano_edificios").join(unicos.drop("_n"), "id").orderBy(F.xxhash64("id")).limit(300)
                 .select("pisos", "Tb", "area_m2", "vs30", "PGA_g", "PGV_cms", "Sa03_g", "Sa06_g", "Sa10_g", "Sa_Tb", "tipologia", "IM_g",
                         "indice_dano", "P_ge_severo").toPandas())
ver_distritos = (del_sismo("sismo_amplificacion_distritos").select("ubigeo", "sa_roca", "pga_roca_g", "pgv_cms", "F_no_lineal", "vs30_mediana")
                 .join(del_sismo("sismo_efecto_distritos").select("ubigeo", "P_ge_severo_medio", "indice_dano_medio"), "ubigeo").toPandas())
ver_distritos["d"] = ver_distritos.ubigeo.map(i_dist)
proceso_lote_spark_s = float(del_sismo("sismo_seguimiento").agg(F.avg("proceso_s")).first()[0])

sdof = [_filtro_sdof(float(T), 1.0 / FS_SIM, AMORTIGUAMIENTO) for T in PERIODOS]
simulador = {
    "generado": pd.Timestamp.now(tz="America/Lima").strftime("%Y-%m-%d %H:%M") + " (hora de Lima)",
    "sismo_base": {"nombre": "Callao 2025", "magnitud": SISMOS_TIEMPO_REAL[0]["magnitud"], "lat": float(ev_real.ev_lat),
                   "lon": float(ev_real.ev_lon), "prof_km": float(ev_real.ev_prof_km), "semilla": SEED},
    "constantes": {"FS_SIM": FS_SIM, "G_CM_S2": G_CM_S2, "VP_KM_S": VP_KM_S, "VS_KM_S": VS_KM_S, "RAZON_P_S": RAZON_P_S, "RUIDO_G": RUIDO_G,
                   "DURACION_MAX_S": DURACION_MAX_S, "LOTE_S": LOTE_S, "STA_S": STA_S, "LTA_S": LTA_S, "UMBRAL_STA_LTA": UMBRAL_STA_LTA,
                   "VENTANA_P_S": VENTANA_P_S, "PASO_P_S": PASO_P_S, "UMBRAL_P_MODELO": UMBRAL_P_MODELO, "RAZON_V_H": RAZON_V_H,
                   "RAZON_PZ_PH": RAZON_PZ_PH, "UMBRAL_SACUDIDA_G": UMBRAL_SACUDIDA_G, "ESCALA_STEAD": ESCALA_STEAD,
                   "PISOS_CONCRETO": PISOS_CONCRETO, "PISOS_MAX_LIVIANO": PISOS_MAX_LIVIANO},
    "sintetico": SINT,
    "filtros": {"senal": filtro_sos(FILTRO_HZ), "p": filtro_sos((1.0, 25.0))},
    "sdof": {"periodos": PERIODOS.tolist(), "b": [s[0].tolist() for s in sdof], "a": [s[1].tolist() for s in sdof]},
    "amplificacion": {"vs30": VS30_GRID.tolist(), "ln_f": ln_f.tolist(), "f4": F4_NL.tolist(), "f5": F5_NL.tolist(), "pga_ref_nl": PGA_REF_NL,
                      "version": int(param["version_amplificacion"]), "algoritmo": param["algoritmo_amplificacion"]},
    "modelo_onda": {**MODELO_ONDA, "version": meta_onda["version"], "algoritmo": meta_onda["algoritmo"], "metricas": meta_onda["metricas"]},
    "modelo_dano": {**MODELO_DANO, "version": meta_dano["version"], "algoritmo": meta_dano["algoritmo"], "metricas": meta_dano["metricas"],
                    "rangos": RANGO_DANO},
    "fragilidad": columnas_json(FRAG.toPandas()), "pisos_max_gem": PISOS_MAX_GEM, "tipologias": tipologias,
    "distritos": {**columnas_json(BASE_DISTRITOS, {"centroide_lon": 6, "centroide_lat": 6, "vs30_mediana": 1}),
                  "contorno": [json.loads(contornos[u]) if contornos.get(u) else None for u in ubigeos]},
    "hex": {"h3": hexes.h3_r8.tolist(), "borde": [[[round(x, 6), round(y, 6)] for x, y in json.loads(b)["coordinates"][0][:-1]] for b in hexes.borde],
            "dS": [None if a is None else [round(float(x), 6) for x in a] for a in hexes.dS_arr]},
    "grupos": columnas_json(grupos[["d", "h", "v", "t", "pisos_gem", "count"]]),
    "muestra_rf": columnas_json(muestra[["d", "h", "v", "t", "vs30", "pisos", "Tb", "area_m2"]], {"vs30": 2, "Tb": 5, "area_m2": 2}),
    "verificacion": {"sismo": SISMO_VERIF, "proceso_lote_spark_s": proceso_lote_spark_s, "lote_s": LOTE_S, "ventanas": columnas_json(ver_ventanas), "edificios": columnas_json(ver_edificios),
                     "distritos": {**columnas_json(ver_distritos.drop(columns=["sa_roca", "F_no_lineal"])),
                                   "sa_roca": [list(map(float, a)) for a in ver_distritos.sa_roca],
                                   "F_no_lineal": [list(map(float, a)) for a in ver_distritos.F_no_lineal]}},
}
os.makedirs(os.path.dirname(SIM_WEB), exist_ok=True)
with open(SIM_WEB + ".tmp", "w", encoding="utf-8") as f:
    json.dump(simulador, f, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
os.replace(SIM_WEB + ".tmp", SIM_WEB)
print(f"Simulador web: {SIM_WEB} ({os.path.getsize(SIM_WEB) / 1e6:.1f} MB) | grupos {len(grupos):,} con {int(grupos['count'].sum()):,} edificios | "
      f"muestra {len(muestra):,} | hexágonos {len(hexes):,} | nodos {sum(len(a['f']) for a in MODELO_ONDA['arboles'])} y {sum(len(a['f']) for a in MODELO_DANO['arboles'])}")

## 11. Métricas

In [ ]:
guardar_metricas("11_sismo_tiempo_real")